# E(3)-Equivariant Diffusion Propagator: Colab Workflow

This notebook runs setup, training, evaluation, and rollout on Google Colab CUDA.

## 1. Check GPU

In Colab, select `Runtime > Change runtime type > T4 GPU` or another GPU first.

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 2. Clone Repository and Install

Replace `REPO_URL` with your GitHub repository URL after uploading this project.

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/diffusion-models.git'
PROJECT_DIR = '/content/diffusion-models'

import os
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!pip install -e .

## 3. Prepare MD Files

Place your files here:

```text
data/argon/nve.tpr
data/argon/nve.trr
```

You can upload from the Colab file browser, mount Google Drive, or copy from Drive in the next cell.

In [ ]:
# Optional: mount Google Drive and copy files from it.
# from google.colab import drive
# drive.mount('/content/drive')

!mkdir -p data/argon
# Example:
# !cp /content/drive/MyDrive/md/argon/nve.tpr data/argon/nve.tpr
# !cp /content/drive/MyDrive/md/argon/nve.trr data/argon/nve.trr

!ls -lh data/argon

## 4. Edit Configuration

`configs/argon_colab.yaml` already expects `data/argon/nve.tpr` and `data/argon/nve.trr`. For a quick smoke test, set `max_frames`, `max_atoms`, and `num_epochs` to small values.

In [ ]:
CONFIG = 'configs/argon_colab.yaml'
!sed -n '1,80p' {CONFIG}

## 5. Train on CUDA

In [ ]:
!python scripts/train.py --config {CONFIG} --device cuda

## 6. One-Step Sample

In [ ]:
!python scripts/sample.py \
  --config {CONFIG} \
  --checkpoint outputs/argon_diffusion/best.pt \
  --frame-index 0 \
  --device cuda

## 7. Evaluate

In [ ]:
!python scripts/evaluate.py \
  --config {CONFIG} \
  --checkpoint outputs/argon_diffusion/best.pt \
  --output-csv outputs/argon_diffusion/evaluation.csv \
  --num-samples 100 \
  --batch-size 4 \
  --device cuda

!head outputs/argon_diffusion/evaluation.csv

## 8. Rollout and Save Prediction Files

In [ ]:
!python scripts/rollout.py \
  --config {CONFIG} \
  --checkpoint outputs/argon_diffusion/best.pt \
  --frame-index 0 \
  --steps 20 \
  --output-prefix outputs/argon_diffusion/rollout \
  --device cuda

!ls -lh outputs/argon_diffusion

## 9. Download Outputs

In [ ]:
from google.colab import files

files.download('outputs/argon_diffusion/best.pt')
files.download('outputs/argon_diffusion/evaluation.csv')
files.download('outputs/argon_diffusion/rollout.npz')
files.download('outputs/argon_diffusion/rollout.xyz')